# H100 Profiling for HPCA Paper

**Purpose**: Capture per-layer performance and memory characteristics for hybrid-AI accelerator workload characterization.

**Models**: 13 configurations across vision, language, dense, and video generation.

**Runtime**: 3-5 hours for full sweep.

---

## Instructions

1. Run cells in order
2. Check GPU availability in Setup section
3. Run Quick Test first (5 min)
4. If test passes, run Full Sweep (3-5 hours)
5. Download results when complete

---

## 1. Setup & Environment Check

In [ ]:
# Check Python and basic imports
import sys
print(f"Python version: {sys.version}")

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch CUDA version: {torch.version.cuda}")

In [ ]:
# Check GPU availability
import torch

print("="*60)
print("GPU CHECK")
print("="*60)
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device count: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Test allocation
    try:
        x = torch.ones(100, 100, device='cuda')
        print(f"\n✓ GPU allocation test: SUCCESS")
        print(f"  Tensor device: {x.device}")
        del x
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"\n✗ GPU allocation test: FAILED")
        print(f"  Error: {e}")
else:
    print("\n⚠️  WARNING: CUDA not available!")
    print("   Check that you're running on a GPU pod.")

print("="*60)

In [ ]:
# Install dependencies (if not already installed)
!pip install -q transformers diffusers accelerate safetensors huggingface-hub
print("✓ Dependencies installed")

In [ ]:
# Optional: Login to HuggingFace for gated models (Llama-3, SDXL, etc.)
# Uncomment and run if you want to use gated models

# from huggingface_hub import login
# login()  # Will prompt for token

# Or provide token directly:
# login(token="your_hf_token_here")

## 2. Import Model Runner

In [ ]:
# Load the model runner
import sys
sys.path.insert(0, '/workspace/h100_profiling')

from run_model import MODEL_RUNNERS

print(f"Available models: {len(MODEL_RUNNERS)}")
for model in MODEL_RUNNERS.keys():
    print(f"  - {model}")

## 3. Quick Test (ResNet-50)

Run this first to verify everything works before the full sweep.

In [ ]:
# Quick test with ResNet-50 (no model download needed)
import torch
import torchvision.models as models
import time

print("="*60)
print("QUICK TEST: ResNet-50")
print("="*60)

try:
    # Load model
    print("Loading ResNet-50...")
    model = models.resnet50(weights='IMAGENET1K_V1')
    model = model.cuda().eval()
    print("✓ Model loaded on GPU")
    
    # Test inference
    print("Running inference...")
    x = torch.randn(1, 3, 224, 224).cuda()
    
    start = time.time()
    with torch.no_grad():
        y = model(x)
    torch.cuda.synchronize()
    elapsed = time.time() - start
    
    print(f"✓ Inference successful!")
    print(f"  Output shape: {y.shape}")
    print(f"  Time: {elapsed*1000:.2f} ms")
    print("\n✓ Quick test PASSED - ready for full sweep!")
    
    # Cleanup
    del model, x, y
    torch.cuda.empty_cache()
    
except Exception as e:
    print(f"\n✗ Quick test FAILED")
    print(f"  Error: {e}")
    import traceback
    traceback.print_exc()

print("="*60)

## 4. PyTorch Profiler Functions

In [ ]:
import torch
import torch.profiler as profiler
import json
import time
from run_model import MODEL_RUNNERS

def profile_model(model_name, batch_size, output_dir="/workspace/results"):
    """
    Profile a model with PyTorch profiler.
    
    Args:
        model_name: Model to profile (from MODEL_RUNNERS)
        batch_size: Batch size to use
        output_dir: Directory to save results
    
    Returns:
        dict: Profiling results
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'='*60}")
    print(f"Profiling: {model_name} @ batch={batch_size}")
    print(f"{'='*60}")
    
    if model_name not in MODEL_RUNNERS:
        raise ValueError(f"Unknown model: {model_name}")
    
    runner = MODEL_RUNNERS[model_name]
    
    # Warmup
    print("  Warmup run...")
    try:
        runner(batch_size)
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        print(f"  ✗ OOM during warmup - batch {batch_size} too large")
        return None
    except Exception as e:
        print(f"  ✗ Error during warmup: {e}")
        return None
    
    # Clear cache
    torch.cuda.empty_cache()
    
    # Profile
    print("  Profiling run...")
    with profiler.profile(
        activities=[profiler.ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
    ) as prof:
        runner(batch_size)
        torch.cuda.synchronize()
    
    # Export trace
    trace_file = f"{output_dir}/trace_{model_name}_b{batch_size}.json"
    prof.export_chrome_trace(trace_file)
    print(f"  ✓ Saved: {trace_file}")
    
    # Extract stats
    events = prof.key_averages()
    results = []
    
    for evt in events:
        if evt.device_type == profiler.DeviceType.CUDA:
            kernel_name = evt.key.lower()
            if any(x in kernel_name for x in ['attention', 'attn', 'qkv', 'softmax']):
                op_class = 'attention'
            elif any(x in kernel_name for x in ['gemm', 'conv', 'matmul', 'linear', 'mlp']):
                op_class = 'dense'
            else:
                op_class = 'other'
            
            results.append({
                'kernel_name': evt.key,
                'op_class': op_class,
                'cuda_time_us': evt.cuda_time_total,
                'count': evt.count,
                'avg_time_us': evt.cuda_time_total / evt.count if evt.count > 0 else 0,
            })
    
    # Save kernels JSON
    kernels_file = f"{output_dir}/kernels_{model_name}_b{batch_size}.json"
    with open(kernels_file, 'w') as f:
        json.dump({
            'model': model_name,
            'batch': batch_size,
            'kernels': results
        }, f, indent=2)
    print(f"  ✓ Saved: {kernels_file}")
    
    # Summary
    total_cuda_time = sum(r['cuda_time_us'] for r in results)
    attention_time = sum(r['cuda_time_us'] for r in results if r['op_class'] == 'attention')
    dense_time = sum(r['cuda_time_us'] for r in results if r['op_class'] == 'dense')
    
    summary = {
        'total_time_ms': total_cuda_time / 1000,
        'attention_time_ms': attention_time / 1000,
        'dense_time_ms': dense_time / 1000,
        'attention_pct': 100 * attention_time / total_cuda_time if total_cuda_time > 0 else 0,
        'dense_pct': 100 * dense_time / total_cuda_time if total_cuda_time > 0 else 0,
    }
    
    print(f"\n  Summary:")
    print(f"    Total: {summary['total_time_ms']:.2f} ms")
    print(f"    Attention: {summary['attention_time_ms']:.2f} ms ({summary['attention_pct']:.1f}%)")
    print(f"    Dense: {summary['dense_time_ms']:.2f} ms ({summary['dense_pct']:.1f}%)")
    
    # Cleanup
    torch.cuda.empty_cache()
    
    return summary

print("✓ Profiling functions loaded")

## 5. Test Single Model (Optional)

Test profiling on one model before running the full sweep.

In [ ]:
# Test profiling on ResNet-50 at batch=1
result = profile_model('resnet-50', batch_size=1)

if result:
    print("\n✓ Single model test PASSED")
else:
    print("\n✗ Single model test FAILED")

## 6. Full Sweep

**WARNING**: This will take 3-5 hours!

Only run this if:
- ✅ Quick test passed
- ✅ Single model test passed
- ✅ You have time to let it run

In [ ]:
# Full sweep configuration
MODELS = [
    "unet-sd",
    "sdxl",
    "dit-xl",
    "unet-3d",
    "llama-8b-1k",
    "llama-8b-4k",
    "llama-8b-16k",
    "llama-8b-64k",
    "resnet-50",
    "cogvideox-16f-480p",
    "cogvideox-49f-480p",
    "cogvideox-81f-480p",
    "cogvideox-49f-720p",
]

BATCHES = [1, 2, 4, 8, 16, 32, 64, 128]

print(f"Total configurations: {len(MODELS)} models × {len(BATCHES)} batches = {len(MODELS) * len(BATCHES)} max")
print(f"Expected runtime: 3-5 hours")
print(f"\nReady to start full sweep!")

In [ ]:
# Run full sweep
import time
from datetime import datetime

print("="*60)
print("STARTING FULL SWEEP")
print("="*60)
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

start_time = time.time()
succeeded = 0
failed = 0
results_summary = []

for model_idx, model in enumerate(MODELS, 1):
    print(f"\n{'='*60}")
    print(f"Model {model_idx}/{len(MODELS)}: {model}")
    print(f"{'='*60}")
    
    for batch_idx, batch in enumerate(BATCHES, 1):
        print(f"\n  [{model_idx}.{batch_idx}] Batch={batch}... ", end="")
        
        try:
            result = profile_model(model, batch)
            
            if result:
                succeeded += 1
                results_summary.append({
                    'model': model,
                    'batch': batch,
                    'status': 'success',
                    **result
                })
                print(f"✓ Success ({result['total_time_ms']:.1f}ms)")
            else:
                failed += 1
                results_summary.append({
                    'model': model,
                    'batch': batch,
                    'status': 'failed'
                })
                print(f"✗ Failed (OOM or error)")
                break  # Stop trying larger batches for this model
                
        except KeyboardInterrupt:
            print("\n\n⚠️  Sweep interrupted by user")
            break
        except Exception as e:
            print(f"✗ Exception: {e}")
            failed += 1
            break
    
    # Save intermediate results
    with open('/workspace/results/summary.json', 'w') as f:
        json.dump(results_summary, f, indent=2)

elapsed = time.time() - start_time
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)

print("\n" + "="*60)
print("SWEEP COMPLETE")
print("="*60)
print(f"Succeeded: {succeeded}")
print(f"Failed: {failed}")
print(f"Total time: {hours}h {minutes}m")
print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nResults saved to: /workspace/results/")
print("="*60)

## 7. View Results Summary

In [ ]:
# Load and display summary
import json
import pandas as pd

with open('/workspace/results/summary.json', 'r') as f:
    summary = json.load(f)

df = pd.DataFrame(summary)
print(f"\nTotal results: {len(df)}")
print(f"Successful: {len(df[df['status'] == 'success'])}")
print(f"Failed: {len(df[df['status'] == 'failed'])}")
print("\n" + "="*80)
print("Results Summary")
print("="*80)
display(df)

In [ ]:
# Check generated files
import os

files = os.listdir('/workspace/results')
traces = [f for f in files if f.startswith('trace_')]
kernels = [f for f in files if f.startswith('kernels_')]

print(f"Generated files:")
print(f"  Traces: {len(traces)}")
print(f"  Kernels: {len(kernels)}")
print(f"  Total: {len(files)} files")

## 8. Download Results

After the sweep completes, download all results to your local machine.

In [ ]:
# Create download archive
!cd /workspace && tar -czf h100_results.tar.gz results/
print("✓ Created archive: /workspace/h100_results.tar.gz")
print("\nDownload with:")
print("  scp '<runpod-ssh>:/workspace/h100_results.tar.gz' ./")
print("\nOr use RunPod's file browser to download it.")

## 9. Stop Pod (When Done)

**IMPORTANT**: Stop the pod to avoid charges!

In [ ]:
# Stop the pod
import os

pod_id = os.environ.get('RUNPOD_POD_ID')
if pod_id:
    print(f"Pod ID: {pod_id}")
    print(f"\nTo stop this pod, run in terminal:")
    print(f"  runpodctl stop pod {pod_id}")
    print(f"\nOr stop it from RunPod UI:")
    print(f"  Pods → Your Pod → STOP")
else:
    print("Could not detect pod ID")
    print("Stop pod manually from RunPod UI")